In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_recall_curve, auc
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

In [ ]:
df = pd.read_csv("../../data/resample_normalized_flagged_v3.csv")

In [ ]:
df = df.drop(columns=["apogee_id", "STARFLAGS"])

In [ ]:
class_columns = ['class_spectral', 'class_lum_logg', 'class_lum_jhk', 'class_lum_bins_logg', 'class_lum_bins_jhk']

In [ ]:
chemical_columns = ['C_FE', 'CI_FE', 'N_FE', 'O_FE', 'NA_FE', 'MG_FE', 'AL_FE', 'SI_FE', 'S_FE', 'K_FE', 'CA_FE', 'TI_FE', 'V_FE', 'CR_FE', 'MN_FE', 'NI_FE', 'FE_H']

In [ ]:
physique_columns = ['J', 'H', 'K', 'LOGG', 'M_H', 'VMICRO', 'VMACRO']

In [ ]:
df_chem = df.drop(columns=physique_columns)

In [ ]:
df_phys = df.drop(columns=chemical_columns)

## Target class_spectral

In [ ]:
target_column = 'class_spectral'
X = df_chem.drop(columns=class_columns)
y = df_chem[target_column]

In [ ]:
# Calcul des fréquences des classes
frequencies = df_chem[target_column].value_counts(normalize=True)

# Calcul des poids inverses
weights = (1 / frequencies).to_dict()

In [ ]:
# Division en ensemble d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
sample_weights = compute_sample_weight(class_weight=weights, y=y_train)

In [ ]:
# Initialisation du modèle
hgb = HistGradientBoostingClassifier(loss="log_loss", learning_rate=0.1, max_iter=100)

In [ ]:
# Entraînement
hgb.fit(X_train, y_train, sample_weight=sample_weights)

In [ ]:
# Prédiction
y_pred = hgb.predict(X_test)

In [ ]:
# Évaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
f1 = f1_score(y_test, y_pred, average="weighted")  # "weighted" pour prendre en compte le déséquilibre
print("F1 Score:", f1)

In [ ]:
y_probs = hgb.predict_proba(X_test)  # Probabilités prédites des classes

# Si c'est un problème binaire :
if len(np.unique(y_test)) == 2:
    precision, recall, _ = precision_recall_curve(y_test, y_probs[:, 1])
    auc_pr = auc(recall, precision)
    print("AUC-PR:", auc_pr)

# Si c'est un problème multiclasse :
else:
    auc_pr_list = []
    for i in range(len(np.unique(y_test))):
        precision, recall, _ = precision_recall_curve((y_test == i).astype(int), y_probs[:, i])
        auc_pr_list.append(auc(recall, precision))
    auc_pr = np.mean(auc_pr_list)  # Moyenne sur toutes les classes
    print("AUC-PR (moyenne sur classes):", auc_pr)

In [ ]:
auc_pr_list